# Build a mini-ACMED in 15 minutes
### From job descriptions to ISIC codes, with a human in the loop

In this notebook you build a small version of **ACMED**, the system NISR uses to code survey
descriptions of economic activity into **ISIC Rev.4**. It uses the same recipe as production:
TF-IDF features → LinearSVC → calibrated confidence → **confidence routing**.

**How to run it:** **Runtime → Run all**. It needs no installation and runs in under a minute.
Then scroll down, and at the end **you set the routing thresholds yourself**.

> **About the data:** 5,000 real Labour Force Survey descriptions, mostly in Kinyarwanda,
> with a little English and French. They are limited to the **20 most common ISIC codes** and
> duplicates are removed. Descriptions that appeared to contain a person's name were
> filtered out before publication. That filtering is itself a governance step.

## 1 · Load the data
Each row is a free-text description as the enumerator wrote it, plus the ISIC code
a human coder assigned. These human-coded rows are what the model learns from.

In [ ]:
#@title Load the data { display-mode: "form" }
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATA_URL = "https://raw.githubusercontent.com/YOUR-ACCOUNT/YOUR-REPO/main/acmed_demo_5000.csv"  # <- replace with the public link
LOCAL = "acmed_demo_5000.csv"
df = pd.read_csv(LOCAL if os.path.exists(LOCAL) else DATA_URL, dtype={"isic_code": str})
df["isic_code"] = df["isic_code"].str.zfill(4)
LABEL = dict(zip(df.isic_code, df.isic_label))

# ====
# Shared chart style
# ====
BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#898781", "#e1e0d9"
TIER_COLORS = {"Auto-coded": "#184f95", "Supervisor review": "#2a78d6", "Manual coding": "#86b6ef"}
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": INK, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

print(f"{len(df):,} descriptions · {df.isic_code.nunique()} ISIC codes\n")
df[["description", "isic_code", "isic_label"]].sample(8, random_state=3)

## 2 · Look before you model
How many examples does each code have? Some codes have ten times more training
examples than others, so the model starts out knowing much more about some activities than others.

In [ ]:
#@title How many examples per ISIC code? { display-mode: "form" }
counts = df.isic_label.value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(counts.index, counts.values, color=BLUE, height=0.7)
for y, v in enumerate(counts.values):
    ax.text(v + 8, y, f"{v:,}", va="center", color=MUTED, fontsize=9)
ax.set_xlabel("Number of descriptions")
ax.set_title("Training examples per ISIC code", loc="left", fontsize=12, color=INK)
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## 3 · Train the model: the whole recipe in about 10 lines
This is the same design as the production system:

* **Word TF-IDF** captures whole words such as *ubucuruzi* (trade) and *kogosha* (to cut hair).
* **Character TF-IDF** captures fragments of words, which makes it robust to misspellings
  (*ubucuruzi*, *ubucuri*, *ubucurizi*) and to Kinyarwanda word forms.
* **LinearSVC** with `class_weight="balanced"` so that rare codes still count.
* **CalibratedClassifierCV** turns the SVM's raw scores into **confidence scores** we can route on.

In [ ]:
#@title Train the model (the code is the lesson, so it stays visible)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# ====
# Hold back 30% as unseen test data
# ====
X_train, X_test, y_train, y_test = train_test_split(
    df.description, df.isic_code, test_size=0.3, stratify=df.isic_code, random_state=42)

# ====
# Features -> classifier -> calibrated confidence
# ====
features = make_union(
    TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
)
model = make_pipeline(
    features,
    CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=0.5), cv=3),
)
model.fit(X_train, y_train)
print(f"Trained on {len(X_train):,} descriptions, testing on {len(X_test):,} it has never seen.")

## 4 · How good is it, and where does it struggle?
**Accuracy** counts every record equally, so common codes dominate it.
**Macro F1** gives every code an equal vote, so weak performance on a rare code cannot hide.
ACMED uses Macro F1 to choose between models.

In [ ]:
#@title Scores and the hardest codes { display-mode: "form" }
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

proba = model.predict_proba(X_test)
pred = model.classes_[proba.argmax(1)]
conf = proba.max(1)
truth = y_test.values

print(f"Accuracy : {accuracy_score(truth, pred):.2f}")
print(f"Macro F1 : {f1_score(truth, pred, average='macro'):.2f}\n")

# ====
# F1 per code
# ====
_, _, f1, _ = precision_recall_fscore_support(truth, pred, labels=model.classes_, zero_division=0)
per_code = pd.Series(f1, index=[LABEL[c] for c in model.classes_]).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(per_code.index, per_code.values, color=BLUE, height=0.7)
for y, v in enumerate(per_code.values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", color=MUTED, fontsize=9)
ax.set_xlim(0, 1.08); ax.set_xlabel("F1 score on unseen data (1.0 = perfect)")
ax.set_title("Some codes are easy, some are contested", loc="left", fontsize=12, color=INK)
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

# ====
# Which codes get mixed up most often
# ====
mix = (pd.DataFrame({"true": truth, "pred": pred}).query("true != pred")
         .value_counts().head(5).reset_index(name="times"))
mix["true"] = mix["true"].map(LABEL); mix["predicted as"] = mix.pop("pred").map(LABEL)
print("Most frequent mix-ups:")
mix[["true", "predicted as", "times"]]

**What to notice:** the weakest codes are the different kinds of *shop*. A general shop, a
food shop and a general shop selling other goods are hard to tell apart from a short description,
and human coders disagree on them too. These are the cases we want a human to look at.

## 5 · Try to fool it
Type a description in **Kinyarwanda, English or French**. The model returns its top 3 codes
with a confidence score, and the routing rule decides who makes the final call.
Try something vague (*"ubucuruzi"*, *"business"*) and something specific.

In [ ]:
#@title Classify your own description { display-mode: "form" }
import ipywidgets as w
from IPython.display import display

AUTO, REVIEW = 0.70, 0.50   # the production thresholds

def route(p, auto=AUTO, review=REVIEW):
    return "Auto-coded" if p >= auto else "Supervisor review" if p >= review else "Manual coding"

def classify(text, show=True):
    p = model.predict_proba([text])[0]
    top = p.argsort()[::-1][:3]
    lines = [f'"{text}"  ->  {route(p[top[0]]).upper()}']
    for i in top:
        lines.append(f"    {model.classes_[i]}  {LABEL[model.classes_[i]]:<30} {p[i]:5.0%}")
    if show: print("\n".join(lines) + "\n")

for example in ["kudoda imyenda", "gusya ibigori", "gucuruza imyenda ya caguwa mu isoko", "ubucuruzi", "maize farmer"]:
    classify(example)

box = w.Text(placeholder="Type a job or business description…", layout=w.Layout(width="70%"))
btn = w.Button(description="Classify", button_style="primary")
out = w.Output()
def on_click(_):
    with out:
        out.clear_output(); classify(box.value.strip()) if box.value.strip() else print("Type something first.")
btn.on_click(on_click); box.on_submit(on_click)
display(w.HBox([box, btn]), out)

**What to notice:**
* *kudoda imyenda* (sewing clothes) and *gusya ibigori* (milling maize) are specific, so they are **auto-coded**.
* *gucuruza imyenda ya caguwa mu isoko* (selling second-hand clothes at the market) gets the right code but only
  moderate confidence, so it goes to **supervisor review**.
* *ubucuruzi* ("trade") is too vague to code, so it goes to **manual coding**, which is the right outcome.
* *maize farmer* is **wrong**, but routing still sends it to a human. The model has seen almost no English,
  so language coverage in the training data is a fairness issue, not a technical detail.

## 6 · You are the regulator: set the thresholds
In production, ACMED auto-codes when confidence is **≥ 70%**, sends **50–70%** to a supervisor,
and sends anything **below 50%** to manual coding. Those numbers are a **policy choice**.
Move the sliders and watch the trade-off between how much gets automated and how many
errors pass through with **no human seeing them**.

*This demo model learned from only 5,000 records, so it is much less confident than the production
system, which was trained on about 390,000. The trade-off works the same way.*

In [ ]:
#@title Confidence routing simulator { display-mode: "form" }
correct = pred == truth

def simulate(auto_threshold=0.70, review_threshold=0.50):
    review_threshold = min(review_threshold, auto_threshold)
    tier = np.where(conf >= auto_threshold, "Auto-coded",
                    np.where(conf >= review_threshold, "Supervisor review", "Manual coding"))
    share = {t: (tier == t).mean() for t in TIER_COLORS}

    # ====
    # One stacked bar: where 10,000 incoming records would go
    # ====
    fig, ax = plt.subplots(figsize=(9, 1.6))
    left = 0
    for t, c in TIER_COLORS.items():
        ax.barh(0, share[t], left=left, color=c, edgecolor="#fcfcfb", linewidth=2, height=0.6)
        if share[t] > 0.08:
            ax.text(left + share[t] / 2, 0, f"{t}\n{share[t]:.0%}", ha="center", va="center",
                    color="white" if t != "Manual coding" else INK, fontsize=9)
        elif share[t] > 0.03:
            ax.text(left + share[t] / 2, 0, f"{share[t]:.0%}", ha="center", va="center",
                    color="white" if t != "Manual coding" else INK, fontsize=9)
        left += share[t]
    ax.set_xlim(0, 1); ax.axis("off")
    ax.set_title(f"Auto-code at ≥ {auto_threshold:.0%} · review at ≥ {review_threshold:.0%}",
                 loc="left", fontsize=11, color=INK)
    plt.show()

    auto = tier == "Auto-coded"
    per10k = 10_000
    wrong_unseen = (auto & ~correct).mean() * per10k
    to_humans = (~auto).mean() * per10k
    acc_auto = correct[auto].mean() if auto.any() else float("nan")
    print(f"Accuracy of auto-coded records : {acc_auto:.0%}")
    print(f"Per 10,000 survey records:")
    print(f"   {auto.mean()*per10k:>6,.0f} coded automatically, of which ~{wrong_unseen:,.0f} are WRONG and no human sees them")
    print(f"   {to_humans:>6,.0f} go to a person (supervisor or manual coder)")

# ====
# A few settings side by side (a static view that also shows in a pre-run copy)
# ====
rows = []
for a, r in [(0.90, 0.70), (0.70, 0.50), (0.50, 0.30), (0.30, 0.10)]:
    auto = conf >= a
    rows.append({"Auto-code ≥": f"{a:.0%}", "Review ≥": f"{r:.0%}", "Automated": f"{auto.mean():.0%}",
                 "Accuracy when automated": f"{correct[auto].mean():.0%}" if auto.any() else "–",
                 "Wrong & unseen per 10k": f"{(auto & ~correct).mean()*10_000:,.0f}"})
display(pd.DataFrame(rows).set_index("Auto-code ≥"))

w.interact(simulate,
           auto_threshold=w.FloatSlider(value=0.70, min=0.30, max=0.95, step=0.05, description="Auto-code ≥",
                                        style={"description_width": "initial"}, readout_format=".0%"),
           review_threshold=w.FloatSlider(value=0.50, min=0.10, max=0.90, step=0.05, description="Review ≥",
                                          style={"description_width": "initial"}, readout_format=".0%"));

## 7 · Your call
**Put your answer in the chat:** *Which auto-code threshold would you approve for official statistics, and why?*

What the production system adds on top of this notebook:
* It is trained on **~390,000** records and **432** codes, not 5,000 records and 20 codes.
* The model is exported to **ONNX** and served from NISR's own infrastructure (aicoder.statistics.gov.rw).
* The **supervisor corrections** are fed back as a signal of where the model is drifting and are used for retraining.

← Back to the presentation: *Optimizing the model for deployment*.